In [0]:
# === Librerías necesarias ===
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F

#customers = pd.read_csv('olist_customers_dataset.csv')#
#orders = pd.read_csv('olist_orders_dataset.csv')#
#payments = pd.read_csv('olist_order_payments_dataset.csv')#

#order_items = pd.read_csv('olist_order_items_dataset.csv')#
#products = pd.read_csv('olist_products_dataset.csv')#
#category_translation = pd.read_csv('product_category_name_translation.csv')#


# ===  Parámetros ===
SHEET_ID1 = "1ntzGdcPmkx9Eqs1-JkX45GW9KdY3gs8Rfx8U2X4JBDU"  # olist_customers_dataset
SHEET_ID2 = "14BAecHRihU1nM0fgJhJhdWXe1sglbk6bHRmp_1pkyYg"  # olist_order_items_dataset
SHEET_ID3 = "1XIguhAzbMWxMEabQtZc0Biew02bSjpuX6XYb8jhCwPg"  # olist_orders_dataset
SHEET_ID4 = "1-h1Q_OB-ImUW2VEFAtYbPWGLPpEArQo9eKLdrVwQoJw"  # olist_products_dataset
SHEET_ID5 = "1Rr7RO5_KDHpwvu1XuuDzFtXJGGwzSIGUaw0GZBRJe-U"  # olist_order_payments_dataset 
SHEET_ID6 = "15U3j0unBN0DSzCZFo6JksuMEunr0tGx3CN_-PpOiN0c"  # product_category_name_translation
SHEET_ID7 = "1IciX5rN7Gz6qFypbXb5sqDJ3B7YpFttO72SI8VXg_Qg"  # olist_order_reviews_dataset


SHEET_NAME1 = "olist_customers_dataset"
SHEET_NAME2 = "olist_order_items_dataset"
SHEET_NAME3 = "olist_orders_dataset"
SHEET_NAME4 = "olist_products_dataset"
SHEET_NAME5 = "olist_order_payments_dataset"
SHEET_NAME6 = "product_category_name_translation"
SHEET_NAME7 = "olist_order_reviews_dataset"

TABLE_NAME1 = "bronce.olist_customers_dataset"
TABLE_NAME2 = "bronce.olist_order_items_dataset"
TABLE_NAME3 = "bronce.olist_orders_dataset"
TABLE_NAME4 = "bronce.olist_products_dataset"
TABLE_NAME5 = "bronce.olist_order_payments_dataset"
TABLE_NAME6 = "bronce.product_category_name_translation"
TABLE_NAME7 = "bronce.olist_order_reviews_dataset"

# ===  URLs públicas de exportación (modo lectura) ===
url1 = f"https://docs.google.com/spreadsheets/d/{SHEET_ID1}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME1}"
url2 = f"https://docs.google.com/spreadsheets/d/{SHEET_ID2}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME2}"
url3 = f"https://docs.google.com/spreadsheets/d/{SHEET_ID3}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME3}"
url4 = f"https://docs.google.com/spreadsheets/d/{SHEET_ID4}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME4}"
url5 = f"https://docs.google.com/spreadsheets/d/{SHEET_ID5}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME5}"
url6 = f"https://docs.google.com/spreadsheets/d/{SHEET_ID6}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME6}"
url7 = f"https://docs.google.com/spreadsheets/d/{SHEET_ID7}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME7}"


# ===  Crear base de datos 'bronce' si no existe ===
spark.sql("CREATE DATABASE IF NOT EXISTS bronce")

# ===  Bucle de ingesta automática ===
for url, sheet_name, table_name in zip([url1, url2, url3, url4, url5, url6, url7],
                                       [SHEET_NAME1, SHEET_NAME2, SHEET_NAME3, SHEET_NAME4, SHEET_NAME5, SHEET_NAME6, SHEET_NAME7],
                                       [TABLE_NAME1, TABLE_NAME2, TABLE_NAME3, TABLE_NAME4, TABLE_NAME5, TABLE_NAME6, TABLE_NAME7]):

    print(f" Iniciando ingesta desde Google Sheet: {sheet_name}")

    try:
        # Leer los datos de la hoja
        df = pd.read_csv(url)
        print(f" Datos leídos correctamente ({len(df)} filas)")
    except Exception as e:
        print(f" Error al leer la hoja {sheet_name}: {e}")
        continue  # pasa al siguiente dataset

    # Convertir a Spark DataFrame
    df_spark = spark.createDataFrame(df)

    # Agregar columna de timestamp de ingesta
    df_spark = df_spark.withColumn("ingestion_timestamp", F.current_timestamp())

    # Escribir en tabla Delta dentro de la base 'bronce'
    df_spark.write.mode("overwrite").format("delta").saveAsTable(table_name)

    print(f" Ingesta completada: {table_name}")
    print(f" Timestamp de ejecución: {datetime.now()}\n")

print(" Ingesta automática completada para todas las hojas.")